# Week 1, Notebook 3: Multi-Layer Networks on Real Data
## Universal Approximation & Depth Experiments

**What you'll build:** A configurable deep network trained on real data (moons/circles datasets).

**Curriculum points:**
- ① Universal Approximation Theorem: width vs depth tradeoff
- ⑦ Capacity ≠ performance: more layers ≠ better
- ② ReLU enables depth

**Time estimate:** 45–60 minutes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
np.random.seed(42)

# ============================================================
# Generate real-ish datasets (no sklearn needed)
# ============================================================
def make_moons(n=300, noise=0.15):
    """Generate two interleaving half-moons."""
    t = np.linspace(0, np.pi, n // 2)
    x1 = np.c_[np.cos(t), np.sin(t)] + np.random.randn(n // 2, 2) * noise
    x2 = np.c_[np.cos(t) + 0.5, -np.sin(t) + 0.5] + np.random.randn(n // 2, 2) * noise
    X = np.vstack([x1, x2])
    y = np.hstack([np.zeros(n // 2), np.ones(n // 2)])
    idx = np.random.permutation(n)
    return X[idx], y[idx]

def make_circles(n=300, noise=0.08):
    """Generate concentric circles."""
    t = np.linspace(0, 2 * np.pi, n // 2)
    x1 = np.c_[np.cos(t) * 0.5, np.sin(t) * 0.5] + np.random.randn(n // 2, 2) * noise
    x2 = np.c_[np.cos(t) * 1.5, np.sin(t) * 1.5] + np.random.randn(n // 2, 2) * noise
    X = np.vstack([x1, x2])
    y = np.hstack([np.zeros(n // 2), np.ones(n // 2)])
    idx = np.random.permutation(n)
    return X[idx], y[idx]

# Generate data
X_moons, y_moons = make_moons(400, noise=0.15)
X_circles, y_circles = make_circles(400, noise=0.08)

# Normalize
for X in [X_moons, X_circles]:
    X -= X.mean(axis=0)
    X /= X.std(axis=0) + 1e-8

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, X, y, title in [(axes[0], X_moons, y_moons, 'Moons'), 
                          (axes[1], X_circles, y_circles, 'Circles')]:
    ax.scatter(X[y==0, 0], X[y==0, 1], c='steelblue', s=20, alpha=0.6, label='Class 0')
    ax.scatter(X[y==1, 0], X[y==1, 1], c='coral', s=20, alpha=0.6, label='Class 1')
    ax.set_title(f'{title} Dataset')
    ax.legend()
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('w1_03_datasets.png', dpi=100, bbox_inches='tight')
plt.show()

## Part 1: Reusable Network Class (improved from Notebook 2)

In [ ]:
# ============================================================
# Improved Neural Network with train/test split and accuracy
# ============================================================

class Layer:
    def __init__(self, n_in, n_out, activation='relu'):
        scale = np.sqrt(2.0 / n_in) if activation == 'relu' else np.sqrt(1.0 / n_in)
        self.W = np.random.randn(n_in, n_out) * scale
        self.b = np.zeros(n_out)
        self.activation = activation
        self.x = self.z = self.dW = self.db = None
    
    def forward(self, x):
        self.x = x
        self.z = x @ self.W + self.b
        if self.activation == 'relu':
            return np.maximum(0, self.z)
        elif self.activation == 'sigmoid':
            return 1.0 / (1.0 + np.exp(-np.clip(self.z, -500, 500)))
        return self.z
    
    def backward(self, da):
        if self.activation == 'relu':
            dz = da * (self.z > 0).astype(float)
        elif self.activation == 'sigmoid':
            s = 1.0 / (1.0 + np.exp(-np.clip(self.z, -500, 500)))
            dz = da * s * (1 - s)
        else:
            dz = da
        m = self.x.shape[0]
        self.dW = (self.x.T @ dz) / m
        self.db = np.mean(dz, axis=0)
        return dz @ self.W.T


class DeepNet:
    def __init__(self, architecture):
        """architecture: list of (n_neurons, activation) tuples, starting from input."""
        self.layers = []
        for i in range(len(architecture) - 1):
            n_in = architecture[i][0]
            n_out, act = architecture[i + 1]
            self.layers.append(Layer(n_in, n_out, act))
    
    def forward(self, X):
        out = X
        for layer in self.layers:
            out = layer.forward(out)
        return out
    
    def backward(self, y_pred, y_true):
        da = 2 * (y_pred - y_true) / y_true.shape[0]
        for layer in reversed(self.layers):
            da = layer.backward(da)
    
    def update(self, lr):
        for layer in self.layers:
            layer.W -= lr * layer.dW
            layer.b -= lr * layer.db
    
    def count_params(self):
        return sum(l.W.size + l.b.size for l in self.layers)
    
    def fit(self, X_train, y_train, X_val, y_val, lr=0.01, epochs=2000):
        history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
        y_tr = y_train.reshape(-1, 1)
        y_vl = y_val.reshape(-1, 1)
        
        for epoch in range(epochs):
            # Forward + backward on training data
            pred_train = self.forward(X_train)
            loss_train = np.mean((pred_train - y_tr) ** 2)
            self.backward(pred_train, y_tr)
            self.update(lr)
            
            # Validation (forward only)
            pred_val = self.forward(X_val)
            loss_val = np.mean((pred_val - y_vl) ** 2)
            
            # Accuracy
            acc_train = np.mean(((pred_train > 0.5).astype(float) == y_tr).astype(float))
            acc_val = np.mean(((pred_val > 0.5).astype(float) == y_vl).astype(float))
            
            history['train_loss'].append(loss_train)
            history['val_loss'].append(loss_val)
            history['train_acc'].append(acc_train)
            history['val_acc'].append(acc_val)
        
        return history

# Train/test split helper
def split_data(X, y, ratio=0.8):
    n = int(len(X) * ratio)
    idx = np.random.permutation(len(X))
    return X[idx[:n]], y[idx[:n]], X[idx[n:]], y[idx[n:]]

print("✓ DeepNet class ready for experiments")

## Part 2: Universal Approximation Theorem — Width Experiment

**Curriculum Point ①:** A single hidden layer can approximate any continuous function...  
but it may need an impractical number of neurons. Let's test this.

In [ ]:
# ============================================================
# EXPERIMENT: How does WIDTH affect learning? (UAT in action)
# ============================================================
X_tr, y_tr, X_val, y_val = split_data(X_moons, y_moons)

widths = [2, 4, 16, 64, 256]
results = {}

fig, axes = plt.subplots(2, len(widths), figsize=(20, 8))

for idx, w in enumerate(widths):
    np.random.seed(42)
    # Single hidden layer with varying width
    arch = [(2, None), (w, 'relu'), (1, 'sigmoid')]
    net = DeepNet(arch)
    
    history = net.fit(X_tr, y_tr, X_val, y_val, lr=0.05, epochs=2000)
    results[w] = history
    
    # Decision boundary
    xx, yy = np.meshgrid(np.linspace(-3, 3, 150), np.linspace(-3, 3, 150))
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = net.forward(grid).reshape(xx.shape)
    
    axes[0, idx].contourf(xx, yy, zz, levels=20, cmap='RdYlBu_r', alpha=0.7)
    axes[0, idx].scatter(X_val[y_val==0, 0], X_val[y_val==0, 1], c='steelblue', s=15, alpha=0.5)
    axes[0, idx].scatter(X_val[y_val==1, 0], X_val[y_val==1, 1], c='coral', s=15, alpha=0.5)
    axes[0, idx].set_title(f'Width={w} ({net.count_params()} params)')
    axes[0, idx].set_xlim(-3, 3)
    axes[0, idx].set_ylim(-3, 3)
    
    axes[1, idx].plot(history['train_acc'], label='Train', alpha=0.8)
    axes[1, idx].plot(history['val_acc'], label='Val', alpha=0.8)
    axes[1, idx].set_ylim(0.4, 1.05)
    axes[1, idx].set_xlabel('Epoch')
    axes[1, idx].set_ylabel('Accuracy')
    axes[1, idx].legend(fontsize=8)
    axes[1, idx].grid(True, alpha=0.2)

plt.suptitle('UNIVERSAL APPROXIMATION: Width vs. Accuracy (1 Hidden Layer)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('w1_03_uat_width.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nFinal validation accuracies:")
for w, h in results.items():
    print(f"  Width {w:3d}: train={h['train_acc'][-1]:.3f}, val={h['val_acc'][-1]:.3f}")

## Part 3: Depth Experiment — Is Deeper Always Better?

**Curriculum Point ⑦:** Capacity ≠ performance. More layers ≠ better model.

In [ ]:
# ============================================================
# EXPERIMENT: How does DEPTH affect learning?
# ============================================================
X_tr, y_tr, X_val, y_val = split_data(X_circles, y_circles)

configs = [
    ("1 layer (wide)",  [(2, None), (32, 'relu'), (1, 'sigmoid')]),
    ("2 layers",        [(2, None), (16, 'relu'), (16, 'relu'), (1, 'sigmoid')]),
    ("4 layers",        [(2, None), (8, 'relu'), (8, 'relu'), (8, 'relu'), (8, 'relu'), (1, 'sigmoid')]),
    ("8 layers",        [(2, None)] + [(8, 'relu')] * 8 + [(1, 'sigmoid')]),
]

fig, axes = plt.subplots(2, len(configs), figsize=(20, 8))

for idx, (name, arch) in enumerate(configs):
    np.random.seed(42)
    net = DeepNet(arch)
    history = net.fit(X_tr, y_tr, X_val, y_val, lr=0.02, epochs=3000)
    
    # Decision boundary
    xx, yy = np.meshgrid(np.linspace(-3, 3, 150), np.linspace(-3, 3, 150))
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = net.forward(grid).reshape(xx.shape)
    
    axes[0, idx].contourf(xx, yy, zz, levels=20, cmap='RdYlBu_r', alpha=0.7)
    axes[0, idx].scatter(X_val[y_val==0, 0], X_val[y_val==0, 1], c='steelblue', s=15, alpha=0.5)
    axes[0, idx].scatter(X_val[y_val==1, 0], X_val[y_val==1, 1], c='coral', s=15, alpha=0.5)
    axes[0, idx].set_title(f'{name}\n({net.count_params()} params)')
    
    axes[1, idx].plot(history['train_loss'], label='Train', alpha=0.8)
    axes[1, idx].plot(history['val_loss'], label='Val', alpha=0.8)
    axes[1, idx].set_xlabel('Epoch')
    axes[1, idx].set_ylabel('Loss')
    axes[1, idx].legend(fontsize=8)
    axes[1, idx].set_yscale('log')
    axes[1, idx].grid(True, alpha=0.2)

plt.suptitle('DEPTH EXPERIMENT: Deeper ≠ Better (without proper engineering)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('w1_03_depth.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n⚡ KEY INSIGHT: The 8-layer network likely performs WORSE than 2 layers.")
print("   Without BatchNorm and proper initialization, gradients vanish in deep networks.")
print("   We'll fix this in Week 2 with PyTorch.")

## ✅ Week 1 Complete — Self-Assessment

### You should now be able to:
- [ ] Build a neural network from scratch (no frameworks)
- [ ] Implement forward pass AND backprop
- [ ] Explain UAT: "One wide layer CAN approximate anything, but depth is more EFFICIENT"
- [ ] Demonstrate that depth without engineering (init, norm) can hurt
- [ ] Identify overfitting by comparing train vs. val curves

### Concept Map Check:
Can you explain these connections?
1. ReLU → better gradient flow → deeper networks become trainable
2. Chain rule → backprop → efficient gradient computation
3. More parameters ≠ better generalization (will explore in Week 2)
4. Networks minimize loss ≠ networks "understand"

## ➡️ Week 2: Move to PyTorch and go DEEP properly
Open `W2_01_PyTorch_First_Network.ipynb`

## Visualizing the Deep Network Architecture

A diagram showing the 2 hidden layers to demonstrate depth vs width representation.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.DiGraph()
layers = {'Input': ['x1', 'x2'], 'Hidden 1': ['h1_1', 'h1_2', 'h1_3', 'h1_4'], 'Hidden 2': ['h2_1', 'h2_2', 'h2_3', 'h2_4'], 'Output': ['y1', 'y2']}
pos = {}
colors = []
for i, (layer, nodes) in enumerate(layers.items()):
    for j, node in enumerate(nodes):
        pos[node] = (i * 2, len(nodes) / 2.0 - j)
        G.add_node(node)
        if 'Input' in layer: colors.append('lightgreen')
        elif 'Hidden' in layer: colors.append('lightblue')
        else: colors.append('salmon')

for u in layers['Input']:
    for v in layers['Hidden 1']: G.add_edge(u, v)
for u in layers['Hidden 1']:
    for v in layers['Hidden 2']: G.add_edge(u, v)
for u in layers['Hidden 2']:
    for v in layers['Output']: G.add_edge(u, v)

plt.figure(figsize=(10, 5))
nx.draw(G, pos, with_labels=True, node_color=colors, node_size=1200, arrowsize=12)
plt.title("Deep Network Architecture (2 Hidden Layers)")

plt.savefig('w1_03_network_arch.png', dpi=100, bbox_inches='tight')
plt.show()
